# ⚽ World Cup 2026 Win Probability — Phase 1: Data Pipeline (v2)

**This is the fresh rebuild.** New structure, football-native from the ground up.

## What this notebook does
```
1. Pull historical football match data from StatsBomb (free, no key)
2. Pull top-5 European league data for training VOLUME
3. Build game-state snapshots with a football-native feature set
4. Label each snapshot with the 3-class outcome (home win / draw / away win)
5. Validate the dataset
6. Save to data/processed/features_v2.parquet
7. Write a first entry to results/RESULTS.md  ← the permanent record
```

## Design decisions locked in from research
- **Small feature set, small model later** — with ~10k snapshots, fewer features generalise better (avoids memorising).
- **Team-strength decays over match time** — confirmed by the NFL win-prob paper. We build the raw features; the model learns the decay.
- **Draws are the hard class** — every football model struggles here. We track draw metrics separately from day one.
- **Add, don't replace** — WC games get *added* to the training pool later, never swapped in alone.

---
### The analogy for this whole phase
Think of building this dataset like **stocking a library before exam season**. The league games are the thousands of general textbooks (lots of volume, broad knowledge). The World Cup games are a small shelf of past exam papers (few, but exactly the thing we'll be tested on). Right now in Phase 1 we're just *filling the shelves* — collecting and organising every book. We don't study yet. We just make sure every book is on the right shelf with a correct label, so that when we train (study) later, nothing is missing or mislabelled.


## Cell 1 — Environment & folder setup

In [19]:
import os, time, json, warnings
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings('ignore')

# ── Project folders (created if missing) ───────────────────────────────────
ROOT       = Path('.')
DATA_RAW   = ROOT / 'data' / 'raw'
DATA_PROC  = ROOT / 'data' / 'processed'
RESULTS    = ROOT / 'results'
for d in (DATA_RAW, DATA_PROC, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

print('Folders ready:')
print(f'  raw      : {DATA_RAW.resolve()}')
print(f'  processed: {DATA_PROC.resolve()}')
print(f'  results  : {RESULTS.resolve()}')


Folders ready:
  raw      : C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\Worldcup project\FIFA WORLDCUP PREDICTION\notebooks\data\raw
  processed: C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\Worldcup project\FIFA WORLDCUP PREDICTION\notebooks\data\processed
  results  : C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\Worldcup project\FIFA WORLDCUP PREDICTION\notebooks\results


## Cell 2 — The feature set (READ THIS before running)

We use **11 features**. Fewer than the old design on purpose — small data wants a small, focused feature set. Here's every one, in plain English:

| # | Feature | What it means | Example |
|---|---------|---------------|---------|
| 0 | `goal_diff` | home goals minus away goals, capped at ±5 | home 2, away 1 → `+1` |
| 1 | `minute_norm` | how far into the match, 0.0 at kickoff → 1.0 at 90' | minute 45 → `0.50` |
| 2 | `is_second_half` | 0 in first half, 1 in second | minute 60 → `1` |
| 3 | `home_rank_norm` | home team FIFA strength, 0 (weak) → 1 (strong) | Brazil → `~0.99` |
| 4 | `away_rank_norm` | away team FIFA strength, same scale | Qatar → `~0.50` |
| 5 | `rank_diff` | home strength minus away strength | strong vs weak → `+0.4` |
| 6 | `is_knockout` | 0 = group game (draw allowed), 1 = knockout | final → `1` |
| 7 | `lead_changes_norm` | how "swingy" the game has been so far | many swaps → higher |
| 8 | `is_neutral_venue` | 1 for World Cup (nobody's true home) | WC game → `1` |
| 9 | `score_state` | 0 = home losing, 1 = level, 2 = home winning | draw → `1` |
| 10 | `strength_x_time` | rank_diff × (1 − minute_norm) — strength that FADES as time runs out | see below |

**Feature 10 is the important one.** It encodes the research finding directly. `rank_diff` is how much stronger the home team is. We multiply it by `(1 − minute_norm)`, which starts at 1.0 (kickoff) and shrinks to 0.0 (final whistle).

Numerical walk-through, every step:
- Say home team is much stronger: `rank_diff = +0.40`.
- **At kickoff:** minute_norm = 0.00, so (1 − 0.00) = 1.00. Feature = 0.40 × 1.00 = **0.40**. Full strength advantage counts.
- **At half-time:** minute_norm = 0.50, so (1 − 0.50) = 0.50. Feature = 0.40 × 0.50 = **0.20**. Advantage now counts half as much.
- **At the 81st minute:** minute_norm = 0.90, so (1 − 0.90) = 0.10. Feature = 0.40 × 0.10 = **0.04**. Advantage barely matters — by now the actual score matters way more.

So the feature literally says: *"being the stronger team is a big deal early, but late in the game it's the scoreboard that talks."* That's the exact behaviour the NFL paper found, and the exact fix we bolted onto the basketball model after the upset trouble — except here it's baked in from birth.


In [20]:
# Feature definition — the single source of truth for column order
FEATURE_COLS = [
    'goal_diff',          # 0
    'minute_norm',        # 1
    'is_second_half',     # 2
    'home_rank_norm',     # 3
    'away_rank_norm',     # 4
    'rank_diff',          # 5
    'is_knockout',        # 6
    'lead_changes_norm',  # 7
    'is_neutral_venue',   # 8
    'score_state',        # 9
    'strength_x_time',    # 10
]
TARGET_COL = 'outcome'  # 0 = away win, 1 = draw, 2 = home win
N_FEATURES = len(FEATURE_COLS)
N_CLASSES  = 3

print(f'{N_FEATURES} features, {N_CLASSES} outcome classes')
print('Order:', FEATURE_COLS)


11 features, 3 outcome classes
Order: ['goal_diff', 'minute_norm', 'is_second_half', 'home_rank_norm', 'away_rank_norm', 'rank_diff', 'is_knockout', 'lead_changes_norm', 'is_neutral_venue', 'score_state', 'strength_x_time']


## Cell 3 — FIFA strength lookup

We need a number for "how good is this team." Full FIFA rankings need a paid feed, so we use a curated table of approximate strength for the teams that appear in our data. Missing teams default to mid-table (rank 50).

`rank_to_norm` turns a rank (1 = best) into a 0–1 strength score. Worked example:
- Rank 1 (best team): (100 − 1) / 99 = 99/99 = **1.00** → maximum strength.
- Rank 50 (average): (100 − 50) / 99 = 50/99 = **0.505** → middle.
- Rank 100 (weak): (100 − 100) / 99 = 0/99 = **0.00** → minimum strength.


In [21]:
# Approximate FIFA strength ranks (lower = stronger). Curated for teams in our data.
FIFA_RANK = {
    'BRA':1,'ARG':2,'FRA':3,'ENG':4,'BEL':5,'NED':6,'POR':7,'ESP':8,'ITA':9,'GER':10,
    'CRO':11,'URU':12,'COL':13,'MEX':14,'USA':15,'SUI':16,'DEN':17,'SEN':18,'WAL':19,'IRN':20,
    'SRB':21,'MAR':22,'PER':23,'JPN':24,'SWE':25,'POL':26,'CHI':27,'KOR':28,'TUN':29,'CRC':30,
    'AUS':31,'NGA':32,'EGY':33,'SCO':34,'NOR':35,'TUR':36,'GHA':37,'ECU':38,'CIV':39,'QAT':40,
    'CAN':41,'CMR':42,'KSA':43,'IRQ':44,'RSA':45,'DZA':46,'CPV':47,'JOR':48,'UZB':49,'NZL':50,
    'CZE':51,'AUT':52,'BIH':53,'COD':54,'PAN':55,'SVK':56,
}
DEFAULT_RANK = 50
MAX_RANK     = 100

def rank_to_norm(rank):
    # rank 1 -> 1.0 (strongest), rank 100 -> 0.0 (weakest)
    return max(0.0, (MAX_RANK - rank) / (MAX_RANK - 1))

def get_strength(code):
    return rank_to_norm(FIFA_RANK.get(code, DEFAULT_RANK))

# Quick check
for t in ['BRA','FRA','QAT','NZL']:
    print(f'  {t}: rank {FIFA_RANK.get(t, DEFAULT_RANK):>3} -> strength {get_strength(t):.3f}')


  BRA: rank   1 -> strength 1.000
  FRA: rank   3 -> strength 0.980
  QAT: rank  40 -> strength 0.606
  NZL: rank  50 -> strength 0.505


## Cell 4 — The snapshot builder

A "snapshot" is a freeze-frame of the match at one moment, turned into our 11 features plus the final result. One match becomes many snapshots (one at kickoff, one after each goal, one every 5 minutes, one at full time).

The analogy: it's like **taking photos at a party at fixed times**. Each photo captures "who's ahead right now, how long's left, how strong each side is." Later the model flips through thousands of these photos, each stamped with how the night ended, and learns which mid-party situations tend to lead to which endings.

> **Update — a leak was found and fixed here.** The `lead_changes_norm` feature used
> to divide by the match's *final* total goal count, which isn't known yet mid-match.
> Worked example: a match ends 3–2 (5 goals total). By minute 20 only 2 goals have
> actually happened, and the lead has changed once. Old (leaky): `1 / 5 = 0.2` — uses
> the future final total. Fixed: `1 / 2 = 0.5` — uses only goals scored *so far*
> (`hs + as_`, the home and away goal counts at that checkpoint). At full-time the two
> versions agree exactly, since "goals so far" and "final total" are then the same
> number — so this only ever affected mid-match snapshots.


In [22]:
def build_snapshots(home_code, away_code, goal_events, is_knockout,
                    is_neutral, match_id, final_home, final_away):
    '''
    goal_events: sorted list of (minute, 'home'|'away') for each goal.
    Returns a list of feature-dict snapshots, each labelled with the final outcome.
    '''
    # Final outcome label (same for every snapshot in this match)
    if   final_home > final_away: outcome = 2   # home win
    elif final_home < final_away: outcome = 0   # away win
    else:                         outcome = 1   # draw

    h_str = get_strength(home_code)
    a_str = get_strength(away_code)
    rank_diff = h_str - a_str

    # Checkpoints: every 5 min, plus exact goal minutes, plus kickoff and full time
    goal_minutes = [m for (m, _) in goal_events]
    checkpoints  = sorted(set([0] + list(range(5, 91, 5)) + goal_minutes + [90]))

    rows = []
    lead_changes = 0
    prev_leader  = 0  # -1 away ahead, 0 level, +1 home ahead

    for minute in checkpoints:
        # Score as of this minute = all goals with minute <= now
        hs = sum(1 for (m, side) in goal_events if m <= minute and side == 'home')
        as_ = sum(1 for (m, side) in goal_events if m <= minute and side == 'away')

        leader = (hs > as_) - (hs < as_)  # +1, 0, or -1
        if leader != prev_leader and leader != 0:
            lead_changes += 1
        prev_leader = leader

        diff        = int(np.clip(hs - as_, -5, 5))
        minute_norm = min(minute / 90.0, 1.0)
        score_state = 0 if diff < 0 else (2 if diff > 0 else 1)
        strength_x_time = rank_diff * (1.0 - minute_norm)

        rows.append({
            'match_id'          : match_id,
            'minute'            : minute,
            'goal_diff'         : diff,
            'minute_norm'       : round(minute_norm, 4),
            'is_second_half'    : 1 if minute > 45 else 0,
            'home_rank_norm'    : round(h_str, 4),
            'away_rank_norm'    : round(a_str, 4),
            'rank_diff'         : round(rank_diff, 4),
            'is_knockout'       : int(is_knockout),
            'lead_changes_norm' : round(lead_changes / max(1, hs + as_), 4),  # FIXED: goals-so-far, not final total
            'is_neutral_venue'  : int(is_neutral),
            'score_state'       : score_state,
            'strength_x_time'   : round(strength_x_time, 4),
            'outcome'           : outcome,
            'home_code'         : home_code,
            'away_code'         : away_code,
        })
    return rows

print('build_snapshots() ready')
# Tiny self-test: strong home team, wins 2-1, one lead change
_test = build_snapshots('BRA','QAT',[(20,'home'),(50,'away'),(70,'home')],
                        is_knockout=0, is_neutral=1, match_id='TEST',
                        final_home=2, final_away=1)
print(f'Test match -> {len(_test)} snapshots, final outcome label = {_test[-1]["outcome"]} (2=home win expected)')
print(f'strength_x_time at kickoff = {_test[0]["strength_x_time"]}, at ~90min = {_test[-1]["strength_x_time"]}')


build_snapshots() ready
Test match -> 19 snapshots, final outcome label = 2 (2=home win expected)
strength_x_time at kickoff = 0.3939, at ~90min = 0.0


## Cell 5 — Pull StatsBomb World Cup data (free, no key)

StatsBomb publishes full World Cup event data on GitHub. We use it for the WC portion. This is the same reliable source from before.


In [23]:
SB_BASE = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'

# StatsBomb competition/season IDs for men's World Cups
SB_WC = [
    {'competition_id': 43, 'season_id': 106, 'label': 'WC 2022'},
    {'competition_id': 43, 'season_id':   3, 'label': 'WC 2018'},
]

def sb_get(path):
    r = requests.get(f'{SB_BASE}/{path}', timeout=20)
    if r.status_code != 200:
        return None
    return r.json()

def pull_sb_matches():
    cache = DATA_RAW / 'sb_wc_matches.json'
    if cache.exists():
        return json.loads(cache.read_text())
    all_m = []
    for c in SB_WC:
        matches = sb_get(f"matches/{c['competition_id']}/{c['season_id']}.json") or []
        print(f"  {c['label']}: {len(matches)} matches")
        all_m.extend(matches)
        time.sleep(0.4)
    cache.write_text(json.dumps(all_m))
    return all_m

sb_matches = pull_sb_matches()
print(f'Total StatsBomb WC matches: {len(sb_matches)}')


Total StatsBomb WC matches: 128


## Cell 6 — Convert StatsBomb matches into snapshots

Each StatsBomb match gives us the teams, the final score, the stage (group vs knockout), and we fetch its event file to get exact goal timings. That feeds straight into `build_snapshots`.


In [24]:
# Map StatsBomb full country names -> our 3-letter codes
SB_NAME = {
    'Brazil':'BRA','Argentina':'ARG','France':'FRA','England':'ENG','Belgium':'BEL',
    'Netherlands':'NED','Portugal':'POR','Spain':'ESP','Italy':'ITA','Germany':'GER',
    'Croatia':'CRO','Uruguay':'URU','Colombia':'COL','Mexico':'MEX','United States':'USA',
    'Switzerland':'SUI','Denmark':'DEN','Senegal':'SEN','Wales':'WAL','Iran':'IRN',
    'Serbia':'SRB','Morocco':'MAR','Peru':'PER','Japan':'JPN','Sweden':'SWE','Poland':'POL',
    'Chile':'CHI','South Korea':'KOR','Korea Republic':'KOR','Tunisia':'TUN','Costa Rica':'CRC',
    'Australia':'AUS','Nigeria':'NGA','Egypt':'EGY','Scotland':'SCO','Ghana':'GHA',
    'Ecuador':'ECU','Qatar':'QAT','Canada':'CAN','Cameroon':'CMR','Saudi Arabia':'KSA',
    'Russia':'RUS','Iceland':'ICE','Panama':'PAN','Costa Rica':'CRC',
}
def name_to_code(n): return SB_NAME.get(n, (n[:3].upper() if n else 'UNK'))

def sb_goal_events(match_id, home_name):
    '''Fetch event file, return sorted (minute, side) for every goal.'''
    cache = DATA_RAW / f'sb_ev_{match_id}.json'
    if cache.exists():
        events = json.loads(cache.read_text())
    else:
        events = sb_get(f'events/{match_id}.json')
        if events is None:
            return None
        cache.write_text(json.dumps(events))
        time.sleep(0.25)
    goals = []
    for e in events:
        if e.get('period', 1) > 2:
            continue  # ignore extra time / pens for the 90-min model
        # A goal in StatsBomb: Shot with outcome Goal, or an Own Goal Against event
        etype = e.get('type', {}).get('name', '')
        minute = int(e.get('minute', 0))
        if etype == 'Shot' and e.get('shot', {}).get('outcome', {}).get('name') == 'Goal':
            side = 'home' if e.get('team', {}).get('name') == home_name else 'away'
            goals.append((minute, side))
        elif etype == 'Own Goal Against':
            # scored FOR the opponent of the team credited
            side = 'away' if e.get('team', {}).get('name') == home_name else 'home'
            goals.append((minute, side))
    return sorted(goals, key=lambda g: g[0])

wc_rows = []
skipped = 0
print('Building WC snapshots (downloading event files, cached after first run)...')
for i, m in enumerate(sb_matches, 1):
    hn = m.get('home_team', {}).get('home_team_name')
    an = m.get('away_team', {}).get('away_team_name')
    fh = m.get('home_score')
    fa = m.get('away_score')
    if hn is None or fh is None:
        skipped += 1; continue
    stage = m.get('competition_stage', {}).get('name', 'Group Stage')
    is_knock = 0 if 'Group' in stage else 1
    goals = sb_goal_events(m['match_id'], hn)
    if goals is None:
        skipped += 1; continue
    wc_rows += build_snapshots(name_to_code(hn), name_to_code(an), goals,
                               is_knockout=is_knock, is_neutral=1,
                               match_id=f"wc_{m['match_id']}",
                               final_home=fh, final_away=fa)
    if i % 20 == 0:
        print(f'  {i}/{len(sb_matches)} matches -> {len(wc_rows)} snapshots so far')

wc_df = pd.DataFrame(wc_rows)
print(f'\nWC snapshots: {len(wc_df):,} from {wc_df.match_id.nunique()} matches (skipped {skipped})')


Building WC snapshots (downloading event files, cached after first run)...
  20/128 matches -> 414 snapshots so far
  40/128 matches -> 835 snapshots so far
  60/128 matches -> 1263 snapshots so far
  80/128 matches -> 1687 snapshots so far
  100/128 matches -> 2107 snapshots so far
  120/128 matches -> 2530 snapshots so far

WC snapshots: 2,693 from 128 matches (skipped 0)


## Cell 7 — League volume via football-data.org

The leagues give us the training *volume*. We reuse the football-data.org key you already have. It reliably returns the two most recent seasons on the free tier — that's plenty of volume for our purposes. If the key is missing or returns nothing, the notebook simply carries on with the WC data only (and tells you).

Put your key in a `.env` file as `FOOTBALL_DATA_API_KEY=...` or paste it directly below.


In [25]:
from os import getenv
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass

FD_KEY  = getenv('FOOTBALL_DATA_API_KEY', '').strip()
FD_BASE = 'https://api.football-data.org/v4'
FD_HEADERS = {'X-Auth-Token': FD_KEY}
LEAGUES = {2021:'Premier League', 2014:'La Liga', 2002:'Bundesliga',
           2019:'Serie A', 2015:'Ligue 1'}
SEASONS = [2023, 2024]  # free tier reliably returns the two most recent

def fd_get(path, params=None):
    r = requests.get(f'{FD_BASE}{path}', headers=FD_HEADERS, params=params, timeout=20)
    if r.status_code == 429:
        print('  rate limited, waiting 60s'); time.sleep(60)
        r = requests.get(f'{FD_BASE}{path}', headers=FD_HEADERS, params=params, timeout=20)
    return r

league_rows = []
if not FD_KEY:
    print('No football-data.org key found — continuing with WC data only.')
else:
    print('Pulling league matches (cached after first run)...')
    for lid, lname in LEAGUES.items():
        for season in SEASONS:
            cache = DATA_RAW / f'fd_{lid}_{season}.json'
            if cache.exists():
                matches = json.loads(cache.read_text())
            else:
                r = fd_get(f'/competitions/{lid}/matches',
                           params={'season': season, 'status':'FINISHED'})
                if r.status_code != 200:
                    print(f'  {lname} {season}: HTTP {r.status_code} (skipping)')
                    matches = []
                else:
                    matches = r.json().get('matches', [])
                    cache.write_text(json.dumps(matches))
                time.sleep(6.5)  # free tier: 10 req/min
            # Convert (leagues have final scores; approximate goals from FT/HT split)
            for m in matches:
                ft = m.get('score', {}).get('fullTime', {})
                ht = m.get('score', {}).get('halfTime', {})
                fh, fa = ft.get('home'), ft.get('away')
                if fh is None: continue
                hh, ha = (ht.get('home') or 0), (ht.get('away') or 0)
                # Approx goal timeline: half-time goals at min 44, rest at min 89
                goals = ([(44,'home')]*hh + [(44,'away')]*ha +
                         [(89,'home')]*(fh-hh) + [(89,'away')]*(fa-ha))
                goals = sorted(goals, key=lambda g:g[0])
                hc = m.get('homeTeam', {}).get('tla') or m.get('homeTeam', {}).get('shortName','UNK')
                ac = m.get('awayTeam', {}).get('tla') or m.get('awayTeam', {}).get('shortName','UNK')
                league_rows += build_snapshots(str(hc)[:3].upper(), str(ac)[:3].upper(),
                                               goals, is_knockout=0, is_neutral=0,
                                               match_id=f"fd_{m.get('id')}",
                                               final_home=fh, final_away=fa)
        print(f'  {lname}: done')

league_df = pd.DataFrame(league_rows) if league_rows else pd.DataFrame()
print(f'\nLeague snapshots: {len(league_df):,}' +
      (f' from {league_df.match_id.nunique()} matches' if len(league_df) else ''))


Pulling league matches (cached after first run)...
  Premier League: done
  La Liga: done
  Bundesliga: done
  Serie A: done
  Ligue 1: done

League snapshots: 71,905 from 3502 matches


## Cell 8 — Combine, validate, and save

In [26]:
# Combine
frames = [df for df in (wc_df, league_df) if len(df)]
combined = pd.concat(frames, ignore_index=True)

# Tag source for later (WC vs league) — useful in diagnostics
combined['is_wc'] = combined['match_id'].str.startswith('wc_').astype(int)

print('='*58)
print('VALIDATION')
print('='*58)
passed = failed = 0
def chk(name, ok, detail=''):
    global passed, failed
    print(f'{"PASS" if ok else "FAIL"}  {name}' + (f'  ({detail})' if detail else ''))
    passed += ok; failed += (not ok)

chk('no nulls in features', combined[FEATURE_COLS].isnull().sum().sum() == 0)
chk('outcome in {0,1,2}', combined[TARGET_COL].isin([0,1,2]).all())
chk('goal_diff within +-5', combined.goal_diff.between(-5,5).all())
chk('minute_norm within 0..1', combined.minute_norm.between(0,1).all())
chk('strength ranks within 0..1',
    combined.home_rank_norm.between(0,1).all() and combined.away_rank_norm.between(0,1).all())
hr = (combined.outcome==2).mean(); dr = (combined.outcome==1).mean(); ar = (combined.outcome==0).mean()
chk('realistic outcome split', 0.35<=hr<=0.60 and 0.10<=dr<=0.40,
    f'home {hr:.1%} draw {dr:.1%} away {ar:.1%}')
chk('enough matches', combined.match_id.nunique() > 300,
    f'{combined.match_id.nunique()} matches')

# strength_x_time sanity: should shrink toward 0 as minute grows (on average)
early = combined[combined.minute_norm < 0.2].strength_x_time.abs().mean()
late  = combined[combined.minute_norm > 0.8].strength_x_time.abs().mean()
chk('strength_x_time decays over time', late < early, f'early {early:.3f} -> late {late:.3f}')

print(f'\nResult: {passed} passed, {failed} failed')

out = DATA_PROC / 'features_v2.parquet'
combined.to_parquet(out, index=False)
print(f'\nSaved: {out}  shape={combined.shape}')


VALIDATION
PASS  no nulls in features
PASS  outcome in {0,1,2}
PASS  goal_diff within +-5
PASS  minute_norm within 0..1
PASS  strength ranks within 0..1
PASS  realistic outcome split  (home 42.8% draw 25.2% away 32.0%)
PASS  enough matches  (3630 matches)
PASS  strength_x_time decays over time  (early 0.015 -> late 0.001)

Result: 8 passed, 0 failed

Saved: data\processed\features_v2.parquet  shape=(74598, 17)


## Cell 9 — Write to the permanent record (`results/RESULTS.md`)

This is the memory-proofing step. Every important run appends a dated entry to `results/RESULTS.md`. Whatever happens to any chat's memory, this file in your repo is the source of truth.


In [27]:
stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
hr = (combined.outcome==2).mean(); dr = (combined.outcome==1).mean(); ar = (combined.outcome==0).mean()

entry = f'''
## Phase 1 — data pipeline ({stamp})
- Total snapshots: {len(combined):,}
- Matches: {combined.match_id.nunique():,}  (WC: {combined[combined.is_wc==1].match_id.nunique()}, league: {combined[combined.is_wc==0].match_id.nunique()})
- Features: {N_FEATURES}  |  Classes: {N_CLASSES}
- Outcome split: home {hr:.1%} / draw {dr:.1%} / away {ar:.1%}
- Saved to: data/processed/features_v2.parquet
'''

results_md = RESULTS / 'RESULTS.md'
header = '# World Cup 2026 Win Probability — Results Log\n\nThe permanent record. Every run appends here.\n'
if not results_md.exists():
    results_md.write_text(header)
with results_md.open('a') as f:
    f.write(entry)

# Also keep a machine-readable metrics row
metrics_csv = RESULTS / 'metrics.csv'
row = pd.DataFrame([{
    'timestamp': stamp, 'phase': 'phase1_data',
    'snapshots': len(combined), 'matches': combined.match_id.nunique(),
    'home_rate': round(hr,4), 'draw_rate': round(dr,4), 'away_rate': round(ar,4),
}])
if metrics_csv.exists():
    row.to_csv(metrics_csv, mode='a', header=False, index=False)
else:
    row.to_csv(metrics_csv, index=False)

print('Appended to results/RESULTS.md and results/metrics.csv')
print(entry)
print('PHASE 1 COMPLETE  ->  next: phase2b_revised_v2.ipynb  (NOT phase2_build_train.ipynb -- that one is the older, less accurate version)')


Appended to results/RESULTS.md and results/metrics.csv

## Phase 1 — data pipeline (2026-07-04 09:47 UTC)
- Total snapshots: 74,598
- Matches: 3,630  (WC: 128, league: 3502)
- Features: 11  |  Classes: 3
- Outcome split: home 42.8% / draw 25.2% / away 32.0%
- Saved to: data/processed/features_v2.parquet

PHASE 1 COMPLETE  ->  next: phase2b_revised_v2.ipynb  (NOT phase2_build_train.ipynb -- that one is the older, less accurate version)
